In [6]:
import numpy as np


test1 = np.zeros((20,8,2))
print(test1.shape)
print(np.stack(test1, axis=1).shape)

(20, 8, 2)
(8, 20, 2)


In [34]:
"""Implementation"""

# Setup Imports
import pandas as pd
import numpy as np
import time
import os

import prediction_handler
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.multioutput import ClassifierChain as skl_cc
from sklearn.multioutput import MultiOutputClassifier

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from skmultilearn.problem_transform import BinaryRelevance

from sklearn.preprocessing import OneHotEncoder

from sklearn import tree
from skmultilearn.ensemble import RakelO

import scipy


from sklearn.model_selection import cross_val_predict

# Baseline Imports

from tabpfn import TabPFNClassifier

from Classifiers import ClassifierChains as cc

from Classifiers import Ensemble as en

from sklearn.metrics import jaccard_score

def main():
    files = [r"./data/PI_DataSet.txt", r"./data/INI_DataSet.txt", r"./data/NRTI_DataSet.txt",
             r"./data/NNRTI_DataSet.txt"]

    for file in files:

        # Reading in and processing high quality File
        df = pd.read_csv(file, sep='\t')

        # removing index and summary column
        df = df.iloc[:, 1:-1]

        # list of current drugs of the dataset
        drugs = [drug for drug in list(df.columns) if not drug.startswith("P")]

        # Filtering out drugs with less than 10 labels present
        unusable_drugs = [drug for drug in drugs if df[drug].count() <= 10]

        if len(unusable_drugs) > 0:
            df.drop(columns=unusable_drugs, inplace=True)

            drugs = [drug for drug in drugs if drug not in unusable_drugs]

        # dropping rows with na labels
        df.dropna(subset=drugs, inplace=True)

        enc = OneHotEncoder(handle_unknown='error')



        X = df.drop(drugs, axis=1)

        enc.fit(X)
        X_trafo = enc.transform(X).toarray()
        Y = prediction_handler.get_classes(df, drugs, mode="binary")

        #clf = TabPFNClassifier()

        multi_target_pfn = cc(TabPFNClassifier, random_state=42)

        use_kfold = False

        folds = 5

        n_jobs = 4

        X_train, X_test, y_train, y_test = train_test_split(X_trafo, Y, test_size=0.33, random_state=42)

        forest = RandomForestClassifier(random_state=42)
        xgb = XGBClassifier(random_state=42)
        lr = LogisticRegression()

        models = [
            #("BR_LR", MultiOutputClassifier(lr, n_jobs=2)),
            #("BR_XGB", MultiOutputClassifier(xgb, n_jobs=2)),
            #("BR_forest", MultiOutputClassifier(forest, n_jobs=2)),
            #("CC_LR", skl_cc(lr, order="random", random_state=42, chain_method="predict_proba")),
            #("CC_xgb", skl_cc(xgb, order="random", random_state=42)),
            #("CC_forest", skl_cc(forest, order="random", random_state=42)),
            ("Rakel_lr", RakelO(base_classifier=lr, base_classifier_require_dense=[True, True], labelset_size=y_train.shape[1] // 4, model_count=6)),
            ("Rakel_xgb", RakelO(base_classifier=xgb,base_classifier_require_dense=[True, True],labelset_size=y_train.shape[1] // 4, model_count=6)),
            ("Rakel_forest", RakelO(base_classifier=forest, base_classifier_require_dense=[True, True], labelset_size=y_train.shape[1] // 4, model_count=6)),
        ]

        #ensemble = en(cc, random_state=42, n_jobs=n_jobs)

        if not use_kfold:

            for name, model in models:
                print()
                model.fit(X_train, y_train)

                y_pred = model.predict(X_test)
                print(type(y_pred))

                if isinstance(y_pred, scipy.sparse._csr.csr_matrix):
                    y_pred = y_pred.todense()

                y_pred_df = pd.DataFrame(y_pred, columns=drugs)

                y_test_df = pd.DataFrame(y_test, columns=drugs)

                #print(np.array(y_pred_proba).shape)

                prediction_handler.save_multilabel(y_pred_df, y_test_df, label=(
            file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" + file.split("/")[-1].split("_")[
        0] + "_" + name), path="./test/")

                if name.startswith("CC"):

                    y_pred_proba = model.predict_proba(X_test)
                    #y_pred_proba_new = np.stack(y_pred_proba, axis=1)
                    #print(y_pred_proba_new.shape)

                    y_pred_proba_new = pd.DataFrame(y_pred_proba, columns = drugs)

                    prediction_handler.save_multilabel(y_pred_proba_new, y_test_df, label=(
                        file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" + file.split("/")[-1].split("_")[
                    0] + "_" + name + "_probabilities"), path="./test/")
                elif name.startswith("Rakel"):
                    pass
                else:
                    y_pred_proba = model.predict_proba(X_test)

                    y_pred_proba_new = y_pred_proba

                    prediction_handler.save_multilabel_proba(y_pred_proba_new, y_test_df, label=(
                            file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" + file.split("/")[-1].split("_")[
                        0] + "_" + name + "_probabilities"), path="./test/")

if __name__ == '__main__':
    main()

C:\Users\flori\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\flori\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

<class 'scipy.sparse._csr.csr_matrix'>

<class 'scipy.sparse._csr.csr_matrix'>

<class 'scipy.sparse._csr.csr_matrix'>



KeyboardInterrupt: 

In [3]:
"""Implementation"""

# Setup Imports
import pandas as pd
import numpy as np
import time
import os

import torch
from DELA.DELAModel import DELAModel
from DELA.utils import init_random_seed, generate_default_config, clear_old_logs
from DELA.dataset import *


import prediction_handler
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.multioutput import ClassifierChain as skl_cc
from sklearn.multioutput import MultiOutputClassifier

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from skmultilearn.problem_transform import BinaryRelevance

from sklearn.preprocessing import OneHotEncoder

from sklearn import tree
from skmultilearn.ensemble import RakelO, RakelD

import scipy

import DELA as dela

from sklearn.model_selection import cross_val_predict

# Baseline Imports

from tabpfn import TabPFNClassifier

from Classifiers import ClassifierChains as cc

from Classifiers import Ensemble as en

from sklearn.metrics import jaccard_score


def main():
    files = [r"./data/PI_DataSet.txt", r"../data/INI_DataSet.txt", r"../data/NRTI_DataSet.txt",r"../data/NNRTI_DataSet.txt"]
    #files = [r"../data/INI_DataSet.txt", r"../data/NRTI_DataSet.txt", r"../data/NNRTI_DataSet.txt"]

    for file in files:

        # Reading in and processing high quality File
        df = pd.read_csv(file, sep='\t')

        # removing index and summary column
        df = df.iloc[:, 1:-1]

        # list of current drugs of the dataset
        drugs = [drug for drug in list(df.columns) if not drug.startswith("P")]

        # Filtering out drugs with less than 10 labels present
        unusable_drugs = [drug for drug in drugs if df[drug].count() <= 10]

        if len(unusable_drugs) > 0:
            df.drop(columns=unusable_drugs, inplace=True)

            drugs = [drug for drug in drugs if drug not in unusable_drugs]

        # dropping rows with na labels
        df.dropna(subset=drugs, inplace=True)

        enc = OneHotEncoder(handle_unknown='error')

        X = df.drop(drugs, axis=1)

        enc.fit(X)
        X_trafo = enc.transform(X).toarray()
        Y = utils.get_classes(df, drugs, mode="binary")

        # clf = TabPFNClassifier()

        multi_target_pfn = cc(TabPFNClassifier, random_state=42)

        use_kfold = True

        folds = 5

        n_jobs = 4

        X_train, X_test, y_train, y_test = train_test_split(X_trafo, Y, test_size=0.33, random_state=42)

        # Setting configurations
        configs = generate_default_config()
        # device params
        configs['use_gpu'] = True
        configs['device'] = torch.device('cuda' if torch.cuda.is_available() and configs['use_gpu'] else 'cpu')
        # training params

        configs['beta'] = 1e-4

        # Loading dataset
        configs['shuffle'] = True

        configs['data_standardizing'] = False

        dataset = eval("corel5k")(configs=configs)

        print(dataset)

        """
        configs['dataset_name'] = dataset.name()

        # Setting architecture params
        configs['model_name'] = 'DELAModel'
        configs['in_features'] = dataset.feat_dim
        configs['num_classes'] = dataset.num_class
        configs['latent_dim'] = args.latent_dim

        # Setting other params
        configs['exp'] = args.exp
        configs['exp_dir'] = os.path.join(configs['model_name'],
                                          configs['exp'],
                                          configs['dataset_name'])
        configs['save_checkpoint_path'] = os.path.join(configs['exp_dir'], 'checkpoint')

        # ensemble = en(cc, random_state=42, n_jobs=n_jobs)

        if not use_kfold:

            for name, model in models:
                print()
                model.fit(X_train, y_train)

                y_pred = model.predict(X_test)
                print(type(y_pred))

                if isinstance(y_pred, scipy.sparse._csr.csr_matrix):
                    y_pred = y_pred.todense()

                y_pred_df = pd.DataFrame(y_pred, columns=drugs)

                y_test_df = pd.DataFrame(y_test, columns=drugs)

                # print(np.array(y_pred_proba).shape)

                utils.save_multilabel(y_pred_df, y_test_df, label=(
                        file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" + file.split("/")[-1].split("_")[
                    0] + "_" + name))

                if name.startswith("CC"):

                    y_pred_proba = model.predict_proba(X_test)
                    # y_pred_proba_new = np.stack(y_pred_proba, axis=1)
                    # print(y_pred_proba_new.shape)

                    y_pred_proba_new = pd.DataFrame(y_pred_proba, columns=drugs)

                    utils.save_multilabel(y_pred_proba_new, y_test_df, label=(
                            file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" +
                            file.split("/")[-1].split("_")[
                                0] + "_" + name + "_probabilities"))
                elif name.startswith("Rakel"):
                    pass
                else:
                    y_pred_proba = model.predict_proba(X_test)

                    y_pred_proba_new = y_pred_proba

                    utils.save_multilabel_proba(y_pred_proba_new, y_test_df, label=(
                            file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" +
                            file.split("/")[-1].split("_")[
                                0] + "_" + name + "_probabilities"))

        else:

            for name, model in models:

                print(name)
                kf = KFold(n_splits=folds, random_state=42, shuffle=True)

                print(X_trafo)
                print(Y)

                y_pred, y_true = utils.cv_predict(model, X_trafo, Y, cv=kf, method="single")

                # if isinstance(y_pred, scipy.sparse._csr.csr_matrix):
                #    y_pred = y_pred.todense()

                y_pred_df = pd.DataFrame(y_pred, columns=drugs)

                y_test_df = pd.DataFrame(y_true, columns=drugs)

                # y_pred_new = (y_pred[..., 1] >= 0.5) * 1.0

                # changed the saving mechanism of classifier chain, new way is better but I don't wanna change my system so gotta convert back again
                # y_pred_new = np.stack(y_pred_new, axis=1)

                # print(y_pred_new.shape)

                # y_pred_df = pd.DataFrame(y_pred_new, columns=drugs)

                kfolds = np.zeros((y_pred.shape[0], 1))

                k = 0

                for _, test in kf.split(X, Y):
                    for i in test:
                        kfolds[i] = k
                    k += 1

                # y_pred_df["kFolds"] = kfolds

                y_test = np.zeros((y_pred[0].shape[0], Y.shape[1]))

                t = 0

                for _, test in kf.split(X, Y):
                    for i in test:
                        # print(i)
                        for j in range(Y.shape[1]):
                            y_test[t, j] = Y.iloc[i, j]
                        t += 1



                # print(np.array(y_pred_proba).shape)

                utils.save_multilabel(y_pred_df, y_test_df, k_folds=kfolds, label=(
                        file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" + file.split("/")[-1].split("_")[
                    0] + "_" + name + "_" + str(folds) + "_fold"))


                if name.startswith("CC"):

                    y_pred_proba = model.predict_proba(X_test)
                    # y_pred_proba_new = np.stack(y_pred_proba, axis=1)
                    # print(y_pred_proba_new.shape)

                    y_pred_proba_new = pd.DataFrame(y_pred_proba, columns=drugs)

                    utils.save_multilabel(y_pred_proba_new, y_test_df, label=(
                            file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" +
                            file.split("/")[-1].split("_")[
                                0] + "_" + name + "_" + str(folds) + "_fold"+ "_probabilities"))
                elif name.startswith("Rakel"):
                    pass
                else:
                    y_pred_proba = model.predict_proba(X_test)

                    y_pred_proba_new = y_pred_proba

                    utils.save_multilabel_proba(y_pred_proba_new, y_test_df, label=(
                            file.split("/")[-1].split("_")[0] + "_results/benchmarkings/" +
                            file.split("/")[-1].split("_")[
                                0] + "_" + name + "_" + str(folds) + "_fold" + "_probabilities"))

        """

if __name__ == '__main__':
    main()

FileNotFoundError: [Errno 2] No such file or directory: './Datasets\\corel5k\\corel5k.mat'

In [4]:
import numpy as np

test3 = np.zeros((1000, 3, 2))

for i, x in enumerate(np.stack(test3, axis=1)):
    print(i, x)



0 [[0. 0.]
 [0. 0.]
 [0. 0.]
 ...
 [0. 0.]
 [0. 0.]
 [0. 0.]]
1 [[0. 0.]
 [0. 0.]
 [0. 0.]
 ...
 [0. 0.]
 [0. 0.]
 [0. 0.]]
2 [[0. 0.]
 [0. 0.]
 [0. 0.]
 ...
 [0. 0.]
 [0. 0.]
 [0. 0.]]


In [22]:
test0 = [np.zeros((3,3)), np.zeros((3,3)), np.zeros((3,3))]



print(np.concatenate(test0))

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


In [3]:
import pandas as pd
from scipy.io import arff

# code
arff_file = arff.loadarff('./data/Other_datasets/genbase.arff')

In [15]:
print(pd.DataFrame(arff_file[0]))

       protein PS00010 PS00011 PS00012 PS00014 PS00017 PS00018 PS00019  \
0    b'O00060'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   
1    b'O00139'   b'NO'   b'NO'   b'NO'   b'NO'  b'YES'   b'NO'   b'NO'   
2    b'O02741'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   
3    b'O08424'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   
4    b'O12984'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   
..         ...     ...     ...     ...     ...     ...     ...     ...   
657  b'Q9Z616'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   
658  b'Q9Z961'   b'NO'   b'NO'   b'NO'   b'NO'  b'YES'   b'NO'   b'NO'   
659  b'Q9ZCH7'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   
660  b'Q9ZDQ3'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   b'NO'   
661  b'Q9ZMB7'   b'NO'   b'NO'   b'NO'   b'NO'  b'YES'   b'NO'   b'NO'   

    PS00020 PS00021  ... PDOC00662 PDOC00018 PDOC50001 PDOC00014 PDOC00750  \
0     b'NO'   b'NO'  ...      b'0

In [8]:
import pandas as pd
from tabicl import TabICLClassifier
from tabpfn import TabPFNClassifier
import data_preprocessing
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
import prediction_handler
from sklearn.model_selection import KFold

folds = 5

X, Y, drugs = data_preprocessing.hq_hiv_loader("./data/INI_DataSet.txt", drop_na=True)

clf = TabICLClassifier(random_state=42)
#clf = TabPFNClassifier(random_state=42)
multi_target_pfn = MultiOutputClassifier(clf, n_jobs=2)

kf = KFold(n_splits=folds, random_state=42, shuffle=True)

y_pred, y_true = prediction_handler.cv_predict(multi_target_pfn, X, Y, cv=kf, mode="single", method="predict_proba")

print(y_pred.shape)

CV 1


ValueError: could not convert string to float: '-'

In [3]:
"""Implementation"""

# Setup Imports
import pandas as pd

import prediction_handler

import data_preprocessing

from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression



from sklearn.preprocessing import OneHotEncoder


from skmultilearn.ensemble import RakelO, RakelD

import scipy




# Baseline Imports

from tabpfn import TabPFNClassifier

from Classifiers import ClassifierChains as cc

from Classifiers import Ensemble as en

from sklearn.metrics import jaccard_score

files = [r"./data/PI_DataSet.txt", r"../data/INI_DataSet.txt", r"../data/NRTI_DataSet.txt",r"../data/NNRTI_DataSet.txt"]

file = files[0]


X, Y, drugs = data_preprocessing.hq_hiv_loader(file, drop_na=True)

enc = OneHotEncoder(handle_unknown='error')


enc.fit(X)
X_trafo = enc.transform(X).toarray()

use_kfold = True

folds = 5


forest = RandomForestClassifier(random_state=42)
xgb = XGBClassifier(random_state=42)
lr = LogisticRegression()



model = RakelD(base_classifier=lr, base_classifier_require_dense=[True, True], labelset_size=2)

print()

X_train, X_test, y_train, y_test = train_test_split(X_trafo, Y, test_size=0.33, random_state=42)


model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(type(y_pred))

print(y_pred)

if isinstance(y_pred, scipy.sparse.spmatrix):
    y_pred = y_pred.todense()

y_pred_df = pd.DataFrame(y_pred, columns=drugs)

y_test_df = pd.DataFrame(y_test, columns=drugs)

y_pre_proba = model.predict_proba(X_test)

print(y_pre_proba)
# print(np.array(y_pred_proba).shape)




C:\Users\flori\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\flori\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

<class 'scipy.sparse._lil.lil_matrix'>
<List of Lists sparse matrix of dtype 'int32'
	with 1103 stored elements and shape (339, 8)>
  Coords	Values
  (2, 1)	1
  (2, 2)	1
  (2, 4)	1
  (2, 5)	1
  (4, 0)	1
  (4, 1)	1
  (4, 2)	1
  (4, 3)	1
  (4, 4)	1
  (4, 5)	1
  (4, 6)	1
  (5, 0)	1
  (5, 1)	1
  (5, 2)	1
  (5, 3)	1
  (5, 4)	1
  (5, 5)	1
  (5, 6)	1
  (5, 7)	1
  (6, 0)	1
  (6, 1)	1
  (6, 2)	1
  (6, 3)	1
  (6, 4)	1
  (6, 5)	1
  :	:
  (330, 1)	1
  (330, 2)	1
  (330, 3)	1
  (330, 4)	1
  (330, 5)	1
  (330, 6)	1
  (330, 7)	1
  (333, 0)	1
  (333, 1)	1
  (333, 2)	1
  (333, 3)	1
  (333, 4)	1
  (333, 5)	1
  (333, 6)	1
  (333, 7)	1
  (334, 1)	1
  (334, 2)	1
  (334, 3)	1
  (334, 4)	1
  (334, 5)	1
  (335, 0)	1
  (335, 1)	1
  (335, 2)	1
  (335, 4)	1
  (335, 5)	1
<List of Lists sparse matrix of dtype 'float64'
	with 2712 stored elements and shape (339, 8)>
  Coords	Values
  (0, 0)	0.008603835985758148
  (0, 1)	0.0030874731213888485
  (0, 2)	0.008589161936233531
  (0, 3)	0.00036000990655745676
  (0, 4)	0.0

In [6]:
import numpy as np
import pandas as pd
import random
from itertools import permutations
import collections
import warnings
from tabpfn import TabPFNClassifier

import prediction_handler

from sklearn.base import BaseEstimator, ClassifierMixin

order = list(range(5))

random.shuffle(order)

print(order)

for est_num, i in enumerate(order):
    print(est_num)
    print(i)
    print()

[3, 4, 1, 2, 0]
0
3

1
4

2
1

3
2

4
0

